# 0 Load the Needed Libs

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data_utils

import time, json, datetime 
from tqdm import tqdm
from typing import Tuple, List

import numpy as np 
import pandas as pd 
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

import logging
import os
import random

import torchvision
import torchvision.transforms.v2 as v2
from matplotlib import pyplot as plt

import torch.utils as utils

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)


In [2]:
print('torch version: ', torch.__version__)
print('cuda is available: ', torch.cuda.is_available())

torch version:  2.11.0+cu126
cuda is available:  True


In [3]:
def seed_everything(seed: int = 42):
    """
    固定所有的随机种子，确保代码和实验结果的完全可复现。
    包括 Python 原生、Numpy、PyTorch(CPU/GPU) 以及 CuDNN 后端。
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    
    # 针对使用 CuDNN 的场景，关闭底层算法的自动寻优，固定随机算子
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
seed_everything(42)
print("全局随机种子已固定为: 42")

全局随机种子已固定为: 42


# 1 Preparation For the Data

## 1.1 Pre-Read the Dateset

In [4]:
# 挂载 Google Drive
# from google.colab import drive # type: ignore
# drive.mount('/content/drive')

In [5]:
# ls ../content/drive/MyDrive

In [7]:
def Process_Data(data_path):
    # --- Step 1: Define feature column names ---
    dense_features = ['I' + str(i) for i in range(1, 14)] # 13 integer/count dense features: I1 to I13
    sparse_features = ['C' + str(i) for i in range(1, 27)] # 26 hashed categorical sparse features: C1 to C26
    col_names = ['label'] + dense_features + sparse_features

    # 定义对应的 Numpy 压缩文件路径
    npz_path = data_path.replace('.txt', '.npz').replace('.csv', '.npz')
    
    if os.path.exists(npz_path):
        print(f"Loading processed data directly from Numpy file: {npz_path}...")
        # 如果存在已经处理好的 npz，使用 np.load 加载
        loaded = np.load(npz_path, allow_pickle=True)
        
        # 将各列重新组装回 DataFrame（这样可以完美保持原本的 int 和 float 数据类型）
        data_dict = {col: loaded[col] for col in col_names}
        data = pd.DataFrame(data_dict)
        
        # 恢复 vocab 字典
        sparse_vocab_size = loaded['vocab'].item()
    else:
        print(f"Raw data processing started from {data_path} (This may take a while)...")
        
        # --- 核心优化 1：使用 chunksize 分块读取以降低峰值内存 ---
        print("Progress: [1~2/4] Reading raw generic TSV file in chunks & Fill NaNs...")
        chunk_list = []
        chunk_size = 500000  # 每次读取 50 万行
        
        # pd.read_csv 的 chunksize 参数会返回一个按块读取的生成器，防止巨大的内存一次性分配
        df_iter = pd.read_csv(data_path, sep='\t', header=None, names=col_names, chunksize=chunk_size)
        
        for chunk in tqdm(df_iter, desc="Reading & Processing Chunks"):
            # 在每个 Chunk 内部进行填补，避免读取全部数据后产生巨型 boolean mask 造成极具毁灭性的内存溢出
            # Dense: 为了统一且高效地分配内存，使用 0 填充
            chunk[dense_features] = chunk[dense_features].fillna(0.0)
            # Sparse: 填充 unknown
            chunk[sparse_features] = chunk[sparse_features].fillna('unknown')
            
            # --- 核心优化 2：精确下转数据类型以继续缩小内存基数 ---
            chunk[dense_features] = chunk[dense_features].astype(np.float32)
            chunk['label'] = chunk['label'].astype(np.float32)
            
            chunk_list.append(chunk)

        print("Concatenating chunks into a single DataFrame...")
        data = pd.concat(chunk_list, ignore_index=True)
        del chunk_list # 手动释放列表引用的内存
        
        # --- Step 4: Normalize dense features to [0, 1] ---
        print("Progress: [3/4] Normalizing dense features vs MinMaxScaler...")
        scaler = MinMaxScaler()
        data[dense_features] = scaler.fit_transform(data[dense_features])
        data[dense_features] = data[dense_features].astype(np.float32)

        # --- Step 5: Encode sparse features ---
        print("Progress: [4/4] Encoding sparse features with pd.factorize (Memory Optimized)...")
        sparse_vocab_size = {}
        for feat in tqdm(sparse_features, desc="Encoding Sparse Features"):
            # 核心优化 3: 弃用极其耗费空间储存的 sklearn.LabelEncoder
            # 采用 Pandas 底层的 factorize 算法不仅速度快 3~5 倍，而且内存占用极低
            data[feat], uniques = pd.factorize(data[feat])
            data[feat] = data[feat].astype(np.int32)
            sparse_vocab_size[feat] = len(uniques)

        # 存为 npz 压缩文件
        print(f"Saving the processed DataFrame to {npz_path} for faster future loading...")
        # 将每一列单独存入，外加 vocab 的 dict
        npz_dict = {col: data[col].values for col in col_names}
        npz_dict['vocab'] = np.array(sparse_vocab_size)
        np.savez_compressed(npz_path, **npz_dict)

    return data, sparse_vocab_size, dense_features, sparse_features

In [17]:
class CriteoDataset(utils.data.Dataset):
    """
    ---
    Note
    ---
    PyTorch Dataset wrapper for the Criteo CTR prediction dataset.

    Calls Process_Data internally to handle missing-value imputation,
    dense feature normalization, and sparse feature label-encoding.
    Each sample is returned as three tensors ready for DeepFM input:
        - label    : float32 scalar (0 or 1)
        - dense_x  : float32 vector of shape [13]  (normalized dense features I1-I13)
        - sparse_x : long    vector of shape [26]  (encoded sparse feature indices C1-C26)
    
    ---
    Args
    ---
        data_path (str): path to the raw Criteo TSV file,
                         e.g. './data/Criteo_small/train.txt'
    ---
    Example
    ---
        >>> dataset = CriteoDataset('./data/Criteo_small/train.txt')
        >>> label, dense_x, sparse_x = dataset[0]
        >>> dense_x.shape, sparse_x.shape
        (torch.Size([13]), torch.Size([26]))
    """

    def __init__(self, data_path: str) -> None:
        # Process the raw file once at construction time so __getitem__
        # only needs a fast DataFrame row-lookup during training
        self.data, self.parse_vocab_size, self.dense_features, self.sparse_features = Process_Data(data_path)

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, index: int):
        row = self.data.iloc[index]

        # Binary click label: 0 or 1
        label = torch.tensor(float(row['label']), dtype=torch.float32)

        # Normalized dense features — float32 for direct MLP input
        dense_x = torch.tensor(
            row[self.dense_features].values.astype(float),
            dtype=torch.float32
        )

        # Encoded sparse features indices — long (int64) for nn.Embedding lookup
        sparse_x = torch.tensor(
            row[self.sparse_features].values.astype(int),
            dtype=torch.long
        )
        return label, dense_x, sparse_x
    
dataset = CriteoDataset('./data/Criteo/train.txt')
label, dense_x, sparse_x = dataset[55]
print("Label: ", label)
print("Dense features: ", dense_x)
print("Sparse features: ", sparse_x)

Raw data processing started from ./data/Criteo/train.txt (This may take a while)...
Progress: [1~2/4] Reading raw generic TSV file in chunks & Fill NaNs...


Reading & Processing Chunks: 0it [00:00, ?it/s]

Reading & Processing Chunks: 92it [08:53,  5.80s/it]


Concatenating chunks into a single DataFrame...
Progress: [3/4] Normalizing dense features vs MinMaxScaler...
Progress: [4/4] Encoding sparse features with pd.factorize (Memory Optimized)...


Encoding Sparse Features: 100%|██████████| 26/26 [02:38<00:00,  6.09s/it]


Saving the processed DataFrame to ./data/Criteo/train.npz for faster future loading...
Label:  tensor(0.)
Dense features:  tensor([1.0390e-03, 1.1642e-05, 4.2725e-04, 0.0000e+00, 1.3385e-06, 0.0000e+00,
        1.0655e-04, 0.0000e+00, 0.0000e+00, 9.0909e-02, 4.3290e-03, 0.0000e+00,
        0.0000e+00])
Sparse features:  tensor([ 2, 12, 50, 46,  0,  0, 52,  7,  0, 36, 50, 50, 49,  2, 52, 49,  0, 12,
        11,  1, 50,  0,  6, 19,  0, 20])


# 2 Build up deepFM networks

### 2.1 Module: Basic Layers and Definitions

#### 2.1.1 Features Class Definition

In [18]:
def get_auto_embedding_dim(num_classes):
    return int(np.floor(6 * np.power(num_classes, 0.25)))


class RandomNormal(object):
    def __init__(self, mean=0.0, std=1.0):
        self.mean = mean
        self.std = std

    def __call__(self, vocab_size, embed_dim, padding_idx=None):
        embed = torch.nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)
        torch.nn.init.normal_(embed.weight, self.mean, self.std)
        if padding_idx is not None:
            torch.nn.init.zeros_(embed.weight[padding_idx])
        return embed


class SequenceFeature(object):
    def __init__(self, name, vocab_size, embed_dim=None, pooling="mean", shared_with=None, padding_idx=None, initializer=RandomNormal(0, 0.0001)):
        self.name = name
        self.vocab_size = vocab_size
        if embed_dim is None:
            self.embed_dim = get_auto_embedding_dim(vocab_size)
        else:
            self.embed_dim = embed_dim
        self.pooling = pooling
        self.shared_with = shared_with
        self.padding_idx = padding_idx
        self.initializer = initializer

    def __repr__(self):
        return f'<SequenceFeature {self.name} with Embedding shape ({self.vocab_size}, {self.embed_dim})>'

    def get_embedding_layer(self):
        if not hasattr(self, 'embed'):
            self.embed = self.initializer(self.vocab_size, self.embed_dim, padding_idx=self.padding_idx)
        return self.embed


class SparseFeature(object):
    def __init__(self, name, vocab_size, embed_dim=None, shared_with=None, padding_idx=None, initializer=RandomNormal(0, 0.0001)):
        self.name = name
        self.vocab_size = vocab_size
        if embed_dim is None:
            self.embed_dim = get_auto_embedding_dim(vocab_size)
        else:
            self.embed_dim = embed_dim
        self.shared_with = shared_with
        self.padding_idx = padding_idx
        self.initializer = initializer

    def __repr__(self):
        return f'<SparseFeature {self.name} with Embedding shape ({self.vocab_size}, {self.embed_dim})>'

    def get_embedding_layer(self):
        if not hasattr(self, 'embed'):
            self.embed = self.initializer(self.vocab_size, self.embed_dim, padding_idx=self.padding_idx)
        return self.embed


class DenseFeature(object):
    def __init__(self, name, embed_dim=1):
        self.name = name
        self.embed_dim = embed_dim

    def __repr__(self):
        return f'<DenseFeature {self.name}>'


#### 2.1.2 Activation Layers Definition

In [19]:
class Dice(nn.Module):
    """The Dice activation function mentioned in the `DIN paper
    https://arxiv.org/abs/1706.06978`
    """

    def __init__(self, epsilon=1e-3):
        super(Dice, self).__init__()
        self.epsilon = epsilon
        self.alpha = nn.Parameter(torch.randn(1))

    def forward(self, x: torch.Tensor):
        # x: N * num_neurons
        avg = x.mean(dim=1)  # N
        avg = avg.unsqueeze(dim=1)  # N * 1
        var = torch.pow(x - avg, 2) + self.epsilon  # N * num_neurons
        var = var.sum(dim=1).unsqueeze(dim=1)  # N * 1

        ps = (x - avg) / torch.sqrt(var)  # N * 1

        ps = nn.Sigmoid()(ps)  # N * 1
        return ps * x + (1 - ps) * self.alpha * x


def activation_layer(act_name):
    """Construct activation layers

    Args:
        act_name: str or nn.Module, name of activation function

    Returns:
        act_layer: activation layer
    """
    if isinstance(act_name, str):
        if act_name.lower() == 'sigmoid':
            act_layer = nn.Sigmoid()
        elif act_name.lower() == 'relu':
            act_layer = nn.ReLU(inplace=True)
        elif act_name.lower() == 'dice':
            act_layer = Dice()
        elif act_name.lower() == 'prelu':
            act_layer = nn.PReLU()
        elif act_name.lower() == "softmax":
            act_layer = nn.Softmax(dim=1)
        elif act_name.lower() == 'leakyrelu':
            act_layer = nn.LeakyReLU()
    elif issubclass(act_name, nn.Module):
        act_layer = act_name()
    else:
        raise NotImplementedError
    return act_layer

#### 2.1.3 Pooling Layers Definition

In [20]:
class ConcatPooling(nn.Module):
    """Keep original sequence embedding shape.

    Shape
    -----
    Input: ``(B, L, D)``  
    Output: ``(B, L, D)``
    """

    def __init__(self):
        super().__init__()

    def forward(self, x, mask=None):
        return x


class AveragePooling(nn.Module):
    """Mean pooling over sequence embeddings.

    Shape
    -----
    Input
        x : ``(B, L, D)``
        mask : ``(B, 1, L)``
    Output
        ``(B, D)``
    """

    def __init__(self):
        super().__init__()

    def forward(self, x, mask=None):
        if mask is None:
            return torch.mean(x, dim=1)
        else:
            sum_pooling_matrix = torch.bmm(mask, x).squeeze(1)
            non_padding_length = mask.sum(dim=-1)
            return sum_pooling_matrix / (non_padding_length.float() + 1e-16)


class SumPooling(nn.Module):
    """
    Sum pooling over sequence embeddings.

    Shape
    -----
    Input
        x : ``(B, L, D)``
        mask : ``(B, 1, L)``
    Output
        ``(B, D)``
    """

    def __init__(self):
        super().__init__()

    def forward(self, x, mask=None):
        if mask is None:
            return torch.sum(x, dim=1)
        else:
            return torch.bmm(mask, x).squeeze(1)

### 2.2 Module: Embedding Layers

In [21]:
class InputMask(nn.Module):
    """Return input masks from features.

    Shape
    -----
    Input
        x : dict
            ``{feature_name: feature_value}``; sequence ``(B, L)``, sparse/dense ``(B,)``.
        features : list or SparseFeature or SequenceFeature
            All elements must be sparse or sequence features.
    Output
        - Sparse: ``(B, num_features)``
        - Sequence: ``(B, num_seq, seq_length)``
    """

    def __init__(self):
        super().__init__()

    def forward(self, x, features):
        mask = []
        if not isinstance(features, list):
            features = [features]
        for fea in features:
            if isinstance(fea, SparseFeature) or isinstance(fea, SequenceFeature):
                if fea.padding_idx is not None:
                    fea_mask = x[fea.name].long() != fea.padding_idx
                else:
                    fea_mask = x[fea.name].long() != -1
                mask.append(fea_mask.unsqueeze(1).float())
            else:
                raise ValueError("Only SparseFeature or SequenceFeature support to get mask.")
        return torch.cat(mask, dim=1)
    

class EmbeddingLayer(nn.Module):
    """General embedding layer.

    Stores per-feature embedding tables in ``embed_dict``.

    Parameters
    ----------
    features : list
        Feature objects to create embedding tables for.

    Shape
    -----
    Input
        x : dict
            ``{feature_name: feature_value}``; sequence values shape ``(B, L)``,
            sparse/dense values shape ``(B,)``.
        features : list
            Feature list for lookup.
        squeeze_dim : bool, default False
            Whether to flatten embeddings.
    Output
        - Dense only: ``(B, num_dense)``.
        - Sparse: ``(B, num_features, embed_dim)`` or flattened.
        - Sequence: same as sparse or ``(B, num_seq, L, embed_dim)`` when ``pooling="concat"``.
        - Mixed: flattened sparse plus dense when ``squeeze_dim=True``.
    """

    def __init__(self, features):
        super().__init__()
        self.features = features
        self.embed_dict = nn.ModuleDict()
        self.n_dense = 0
        self.input_mask = InputMask()

        for fea in features:
            if fea.name in self.embed_dict:  # exist
                continue
            if isinstance(fea, SparseFeature) and fea.shared_with is None:
                self.embed_dict[fea.name] = fea.get_embedding_layer()
            elif isinstance(fea, SequenceFeature) and fea.shared_with is None:
                self.embed_dict[fea.name] = fea.get_embedding_layer()
            elif isinstance(fea, DenseFeature):
                self.n_dense += 1

    def forward(self, x, features, squeeze_dim=False):
        sparse_emb, dense_values = [], []
        sparse_exists, dense_exists = False, False
        for fea in features:
            if isinstance(fea, SparseFeature):
                if fea.shared_with is None:
                    sparse_emb.append(self.embed_dict[fea.name](x[fea.name].long()).unsqueeze(1))
                else:
                    sparse_emb.append(self.embed_dict[fea.shared_with](x[fea.name].long()).unsqueeze(1))
            elif isinstance(fea, SequenceFeature):
                if fea.pooling == "sum":
                    pooling_layer = SumPooling()
                elif fea.pooling == "mean":
                    pooling_layer = AveragePooling()
                elif fea.pooling == "concat":
                    pooling_layer = ConcatPooling()
                else:
                    raise ValueError("Sequence pooling method supports only pooling in %s, got %s." % (["sum", "mean"], fea.pooling))
                fea_mask = self.input_mask(x, fea)
                if fea.shared_with is None:
                    sparse_emb.append(pooling_layer(self.embed_dict[fea.name](x[fea.name].long()), fea_mask).unsqueeze(1))
                else:
                    sparse_emb.append(pooling_layer(self.embed_dict[fea.shared_with](x[fea.name].long()), fea_mask).unsqueeze(1))  # shared specific sparse feature embedding
            else:
                dense_values.append(x[fea.name].float() if x[fea.name].float().dim() > 1 else x[fea.name].float().unsqueeze(1))  # .unsqueeze(1).unsqueeze(1)

        if len(dense_values) > 0:
            dense_exists = True
            dense_values = torch.cat(dense_values, dim=1)
        if len(sparse_emb) > 0:
            sparse_exists = True
            # TODO: support concat dynamic embed_dim in dim 2
            # [batch_size, num_features, embed_dim]
            sparse_emb = torch.cat(sparse_emb, dim=1)

        if squeeze_dim:  # Note: if the emb_dim of sparse features is different, we must squeeze_dim
            if dense_exists and not sparse_exists:  # only input dense features
                return dense_values
            elif not dense_exists and sparse_exists:
                # squeeze dim to : [batch_size, num_features*embed_dim]
                return sparse_emb.flatten(start_dim=1) # type: ignore
            elif dense_exists and sparse_exists:
                # concat dense value with sparse embedding
                return torch.cat((sparse_emb.flatten(start_dim=1), dense_values), dim=1) # type: ignore
            else:
                raise ValueError("The input features can note be empty")
        else:
            if sparse_exists:
                return sparse_emb  # [batch_size, num_features, embed_dim]
            else:
                raise ValueError("If keep the original shape:[batch_size, num_features, embed_dim], expected %s in feature list, got %s" % ("SparseFeatures", features))

### 2.3 Module: Logistic Regression

In [22]:
class LR(nn.Module):
    """Logistic regression module.

    Parameters
    ----------
    input_dim : int
        Input dimension.
    sigmoid : bool, default False
        Apply sigmoid to output when True.

    Shape
    -----
    Input: ``(B, input_dim)``
    Output: ``(B, 1)``
    """

    def __init__(self, input_dim, sigmoid=False):
        super().__init__()
        self.sigmoid = sigmoid
        self.fc = nn.Linear(input_dim, 1, bias=True)

    def forward(self, x):
        if self.sigmoid:
            return torch.sigmoid(self.fc(x))
        else:
            return self.fc(x)

### 2.4 Module: Factorization Machine

In [23]:
class FM(nn.Module):
    """Factorization Machine for 2nd-order interactions.

    Parameters
    ----------
    reduce_sum : bool, default True
        Sum over embed dim (inner product) when True; otherwise keep dim.

    Shape
    -----
    Input: ``(B, num_features, embed_dim)``  
    Output: ``(B, 1)`` or ``(B, embed_dim)``
    """

    def __init__(self, reduce_sum=True):
        super().__init__()
        self.reduce_sum = reduce_sum

    def forward(self, x):
        square_of_sum = torch.sum(x, dim=1)**2
        sum_of_square = torch.sum(x**2, dim=1)
        ix = square_of_sum - sum_of_square
        if self.reduce_sum:
            ix = torch.sum(ix, dim=1, keepdim=True)
        return 0.5 * ix

### 2.5 Module: Multi-layer Perceptron

In [24]:
class MLP(nn.Module):
    """Multi-layer perceptron with BN/activation/dropout per linear layer.

    Parameters
    ----------
    input_dim : int
        Input dimension of the first linear layer.
    output_layer : bool, default True
        If True, append a final Linear(*,1).
    dims : list, default []
        Hidden layer sizes.
    dropout : float, default 0
        Dropout probability.
    activation : str, default 'relu'
        Activation function (sigmoid, relu, prelu, dice, softmax).

    Shape
    -----
    Input: ``(B, input_dim)``  
    Output: ``(B, 1)`` or ``(B, dims[-1])``
    """

    def __init__(self, input_dim, output_layer=True, dims=None, dropout=0, activation="relu"):
        super().__init__()
        if dims is None:
            dims = []
        layers = list()
        for i_dim in dims:
            layers.append(nn.Linear(input_dim, i_dim))
            layers.append(nn.BatchNorm1d(i_dim))
            layers.append(activation_layer(activation))
            layers.append(nn.Dropout(p=dropout))
            input_dim = i_dim
        if output_layer:
            layers.append(nn.Linear(input_dim, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, x):
        return self.mlp(x)

### 2.6 Construct DeepFM Model

In [25]:
class DeepFM(torch.nn.Module):
    """
    Deep Factorization Machine Model

    Args:
        deep_features (list): the list of `Feature Class`, training by the deep part module.
        fm_features (list): the list of `Feature Class`, training by the fm part module.
        mlp_params (dict): the params of the last MLP module, keys include:
            `{"dims":list, "activation":str, "dropout":float, "output_layer":bool`}
    """

    def __init__(self, deep_features, fm_features, mlp_params):
        super(DeepFM, self).__init__()
        self.deep_features = deep_features
        self.fm_features = fm_features
        
        self.deep_dims = sum([fea.embed_dim for fea in deep_features])
        self.fm_dims = sum([fea.embed_dim for fea in fm_features])
        
        self.linear = LR(self.fm_dims)  # 1-odrder interaction
        self.fm = FM(reduce_sum=True)  # 2-odrder interaction
        self.embedding = EmbeddingLayer(deep_features + fm_features)
        self.mlp = MLP(self.deep_dims, **mlp_params)

    def forward(self, x):
        input_deep = self.embedding(x, self.deep_features, squeeze_dim=True)  # [batch_size, deep_dims]
        # [batch_size, num_fields, embed_dim]
        input_fm = self.embedding(x, self.fm_features, squeeze_dim=False)

        y_linear = self.linear(input_fm.flatten(start_dim=1))
        y_fm = self.fm(input_fm)
        y_deep = self.mlp(input_deep)  # [batch_size, 1]
        y = y_linear + y_fm + y_deep
        return torch.sigmoid(y.squeeze(1))

# 3 Start Training

## 3.1 Define Train Function

In [28]:
def train(model: nn.Module, loader: data_utils.DataLoader,
                    criterion: nn.Module, optimizer: optim.Optimizer,
                    device: torch.device, epoch: int,) -> float:
    """
    Run one full pass over the training set.

    Returns:
        float: mean BCE loss across all mini-batches.
    """
    model.train()
    total_loss = 0.0

    # leave=False: bar disappears after the epoch finishes, keeping the log clean
    pbar = tqdm(loader, desc='  Train', leave=False, unit='batch')
    for step, (x, labels) in enumerate(pbar):
        x      = {k: v.to(device) for k, v in x.items()}
        labels = labels.to(device)

        optimizer.zero_grad()
        preds = model(x)                   # [B], already sigmoid-activated
        loss  = criterion(preds, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        
        # 记录当前 step 的 loss，仅输出到文件
        file_only_logger.info(f"Epoch [{epoch}] Step [{step}/{len(loader)}] Train Loss: {loss.item():.4f}")
        
        # Update the postfix so the current batch loss is visible in real time
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    return total_loss / len(loader)

## 3.2 Define Evaluate Function

In [29]:
@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: data_utils.DataLoader,
    device: torch.device,
) -> tuple[float, float, float, float, float, float]:
    """
    Compute AUC, LogLoss, Accuracy, Precision, Recall, and F1-Score.

    Returns:
        tuple: (AUC, LogLoss, Accuracy, Precision, Recall, F1-Score).
    """
    model.eval()
    all_preds:  list[float] = []
    all_labels: list[float] = []

    for x, labels in tqdm(loader, desc='  Eval ', leave=False, unit='batch'):
        x     = {k: v.to(device) for k, v in x.items()}
        preds = model(x).cpu()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

    auc = roc_auc_score(all_labels, all_preds)
    logloss = log_loss(all_labels, all_preds)
    
    bin_preds = [1 if p >= 0.5 else 0 for p in all_preds]
    acc = accuracy_score(all_labels, bin_preds)
    pre = precision_score(all_labels, bin_preds, zero_division=0)
    rec = recall_score(all_labels, bin_preds, zero_division=0)
    f1  = f1_score(all_labels, bin_preds, zero_division=0)
    
    return auc, logloss, acc, pre, rec, f1

## 3.3 Define the Hyperparameters

In [ ]:
# ---- Hyperparameters ------------------------------------------ #
train_path    = './data/Criteo/train.txt'
ckpt_dir      = './checkpoints'
logger_dir    = './logs'
batch_size    = 1024        # 8GB VRAM 比较充裕，可以提升到 2048 或 4096 提高 GPU 吞吐
num_epochs    = 10          # 轮数稍增加
learning_rate = 1e-3
weight_decay  = 1e-5        # [新增] 防止过拟合的权重衰减
embed_dim     = 32          # 典型的 CTR Embedding 维度推荐用 16-64 之间
mlp_dims      = [256, 128, 64] # 加深了网络
dropout       = 0.5        # 根据刚才的严重过拟合，提高到 0.3
val_ratio     = 0.1         # fraction of training data held out for validation
early_stop_patience = 3     # stop if no AUC improvement for N epochs
early_stop_min_delta = 1e-4 # minimum AUC gain to reset patience
device        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 3.4 Define Loggers

In [37]:
# 创建 logger 文件夹并生成带时间戳的日志文件名
os.makedirs(logger_dir, exist_ok=True)
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
log_file = os.path.join('./logger', f'{timestamp}.log')

# Configure a module-level logger with timestamp + level + message layout.
# basicConfig is called once here so any handler added by the caller is not
# overridden; if the root logger already has handlers this call is a no-op.
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler(log_file, encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

file_only_logger = logging.getLogger('file_only')
file_only_logger.setLevel(logging.INFO)
file_only_handler = logging.FileHandler(log_file, encoding='utf-8')
file_only_handler.setFormatter(logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s', datefmt='%Y-%m-%d %H:%M:%S'))
file_only_logger.addHandler(file_only_handler)
file_only_logger.propagate = False


## 3.5 Define the Main Function

In [38]:
def build_collate_fn(dense_features: list, sparse_features: list):
    """
    Build a collate function for DataLoader.

    CriteoDataset.__getitem__ returns (label, dense_x, sparse_x) where
    dense_x/sparse_x are flat tensors indexed by position.  torch_rechub's
    EmbeddingLayer expects a dict keyed by feature name.  This function
    converts between the two formats at batch-collation time.

    Args:
        dense_features  (list[str]): ordered dense column names, e.g. ['I1',...,'I13']
        sparse_features (list[str]): ordered sparse column names, e.g. ['C1',...,'C26']

    Returns:
        callable: collate_fn suitable for torch.utils.data.DataLoader
    """
    def _collate(batch):
        labels   = torch.stack([item[0] for item in batch])   # [B]
        dense_x  = torch.stack([item[1] for item in batch])   # [B, 13]
        sparse_x = torch.stack([item[2] for item in batch])   # [B, 26]

        # Unpack each column into a named entry; the model indexes by name
        x: dict[str, torch.Tensor] = {}
        for i, name in enumerate(dense_features):
            x[name] = dense_x[:, i]    # [B], float32
        for i, name in enumerate(sparse_features):
            x[name] = sparse_x[:, i]   # [B], long

        return x, labels

    return _collate

In [ ]:
def main():
    os.makedirs(ckpt_dir, exist_ok=True)

    # ---- Dataset -------------------------------------------------- #
    # IMPORTANT: load the full file once and split afterwards.
    # Loading train/val from separate files would run LabelEncoder
    # independently on each file, so the same hash string could receive
    # a different integer index in val — corrupting the embedding lookup.
    # Splitting a single dataset guarantees both subsets share the same
    # encoder mapping fitted in Process_Data.
    logger.info('Loading and processing dataset...')
    dataset = CriteoDataset(train_path)

    dense_feas  = dataset.dense_features     # ['I1', ..., 'I13']
    sparse_feas = dataset.sparse_features    # ['C1', ..., 'C26']
    vocab       = dataset.parse_vocab_size  # {name: num_unique_values}

    val_size   = int(len(dataset) * val_ratio)
    train_size = len(dataset) - val_size
    train_set, val_set = data_utils.random_split(
        dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)   # reproducible split
    )

    collate_fn = build_collate_fn(dense_feas, sparse_feas)

    # num_workers=0: avoids multiprocess pickling issues on Windows;
    # pin_memory speeds up host→GPU transfers when CUDA is available
    train_loader = data_utils.DataLoader(
        train_set,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=(device.type == 'cuda'),
        collate_fn=collate_fn,
    )
    val_loader = data_utils.DataLoader(
        val_set,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == 'cuda'),
        collate_fn=collate_fn,
    )

    # ---- Feature definitions -------------------------------------- #
    # DenseFeature  : passes the scalar value directly into the MLP
    # SparseFeature : looks up a learned embed_dim-D embedding vector
    dense_feature_objs  = [DenseFeature(name) for name in dense_feas]
    sparse_feature_objs = [
        SparseFeature(name, vocab_size=vocab[name], embed_dim=embed_dim)
        for name in sparse_feas
    ]

    # ---- Model ---------------------------------------------------- #
    # FM   part : sparse features only — captures 2nd-order pairwise interactions
    # Deep part : dense + sparse — captures higher-order non-linear interactions
    model = DeepFM(
        deep_features=dense_feature_objs + sparse_feature_objs,
        fm_features=sparse_feature_objs,
        mlp_params={
            'dims':       mlp_dims,
            'dropout':    dropout,
            'activation': 'relu',
        },
    ).to(device)

    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # Halve LR after 2 epochs without AUC improvement; avoids manual decay tuning
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=2, factor=0.5
    )

    logger.info(
        f'Device : {device} Train  : {train_size:,} samples Val    : {val_size:,} samples'
    )

    # ---- Training loop -------------------------------------------- #
    best_auc = float('-inf')
    no_improve_epochs = 0
    for epoch in range(1, num_epochs + 1):
        train_loss = train(model, train_loader, criterion, optimizer, device, epoch)
        val_auc, val_logloss, val_acc, val_pre, val_rec, val_f1 = evaluate(model, val_loader, device)
        scheduler.step(val_auc)

        lr_now = optimizer.param_groups[0]['lr']
        logger.info(
            f'Epoch [{epoch:>2}/{num_epochs}] | '
            f'Train Loss: {train_loss:.4f} | '
            f'Val AUC: {val_auc:.4f} | '
            f'LogLoss: {val_logloss:.4f} | '
            f'ACC: {val_acc:.4f} | '
            f'Pre: {val_pre:.4f} | '
            f'Rec: {val_rec:.4f} | '
            f'F1: {val_f1:.4f} | '
            f'LR: {lr_now:.2e}'
        )

        # Persist checkpoint whenever validation AUC improves
        if val_auc > best_auc + early_stop_min_delta:
            best_auc  = val_auc
            no_improve_epochs = 0
            ckpt_path = os.path.join(ckpt_dir, 'deepfm_best.pth')
            torch.save(model.state_dict(), ckpt_path)
            logger.info(f'=> Best model saved  (AUC={best_auc:.4f}, LogLoss={val_logloss:.4f}, F1={val_f1:.4f})')
        else:
            no_improve_epochs += 1
            logger.info(f'=> No AUC improvement for {no_improve_epochs} epoch(s)')
            if no_improve_epochs >= early_stop_patience:
                logger.info(f'=> Early stopping triggered after {early_stop_patience} epochs without improvement')
                break

    logger.info(f'\nTraining complete.  Best Val AUC: {best_auc:.4f}')

## 3.6 Start Training

In [ ]:
main()